In [ ]:
!pip install -q ultralytics

In [ ]:
import os, glob, json, time, shutil, zipfile, random
from collections import Counter

import yaml
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
N_GPU = torch.cuda.device_count()
print("GPU count:", N_GPU)
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB")

In [ ]:
PROBE_ONLY   = False   

BASE_MODEL   = "yolo11s.pt"   
EPOCHS       = 35
IMGSZ        = 640
PATIENCE     = 15            

BATCH        = 32 if torch.cuda.device_count() >= 2 else 16
WORKERS      = 2             
DEVICE       = [0, 1] if torch.cuda.device_count() >= 2 else 0

FAST_VAL_SIZE = 2500

DROP_CLASSES = ["machinery", "vehicle"]

WORK       = "/kaggle/working"
UNIFIED    = f"{WORK}/ppe_unified"
RUN_PROJECT = f"{WORK}/ppe_runs"
RUN_NAME    = "phase3_custom_v2"

TIME_BUDGET_HOURS = 9.0 

cfg = dict(base_model=BASE_MODEL, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
           workers=WORKERS, device=str(DEVICE), patience=PATIENCE, seed=SEED,
           fast_val_size=FAST_VAL_SIZE, time_budget_hours=TIME_BUDGET_HOURS,
           drop_classes=DROP_CLASSES, mosaic=0.5)
print(json.dumps(cfg, indent=2))

In [ ]:
def find_phase2_zip():
    
    patterns = [
        "/kaggle/input/**/ppe_unified.zip",
        "/kaggle/input/**/_output_.zip",
    ]
    hits = []
    for pat in patterns:
        hits += glob.glob(pat, recursive=True)
    return hits

if os.path.exists(f"{UNIFIED}/data.yaml"):
    print("ppe_unified already present in /kaggle/working — skipping extraction")
else:
    hits = find_phase2_zip()
    assert hits, ("No Phase 2 zip found. Attach your ppe-2-0 notebook output as an "
                  "input to this notebook (Add Input -> Notebook Output).")
    ZIP_PATH = hits[0]
    print("extracting:", ZIP_PATH)
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(WORK)
    print(f"extracted in {time.time()-t0:.0f}s")

assert os.path.exists(f"{UNIFIED}/data.yaml"), "data.yaml missing after extraction"

with open(f"{UNIFIED}/data.yaml") as f:
    data_cfg = yaml.safe_load(f)

data_cfg["path"] = UNIFIED
with open(f"{UNIFIED}/data.yaml", "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

MASTER = list(data_cfg["names"].values()) if isinstance(data_cfg["names"], dict) else list(data_cfg["names"])
NC = len(MASTER)

for split in ["train", "valid", "test"]:
    d = f"{UNIFIED}/{split}/images"
    print(f"{split:6s}: {len(os.listdir(d)) if os.path.isdir(d) else 0:6d} images")
print("classes:", NC, MASTER)

In [ ]:
KEEP = [c for c in MASTER if c not in DROP_CLASSES]
OLD2NEW = {MASTER.index(c): i for i, c in enumerate(KEEP)}
NC10 = len(KEEP)
print("keeping", NC10, "classes:", KEEP)
print("dropping:", DROP_CLASSES)

MARKER = f"{UNIFIED}/.remapped_10cls"

if os.path.exists(MARKER):
    print("labels already remapped — skipping")
else:
    t0 = time.time()
    for split in ["train", "valid", "test"]:
        lbl_dir = f"{UNIFIED}/{split}/labels"
        bak_dir = f"{UNIFIED}/{split}/labels_orig12"
        
        if not os.path.exists(bak_dir):
            shutil.copytree(lbl_dir, bak_dir)

        kept = dropped = emptied = 0
        for lf in glob.glob(f"{bak_dir}/*.txt"):
            out = []
            with open(lf) as f:
                for line in f:
                    parts = line.split()
                    if len(parts) < 5:
                        continue
                    c = int(parts[0])
                    if c in OLD2NEW:
                        out.append(" ".join([str(OLD2NEW[c])] + parts[1:5]))
                        kept += 1
                    else:
                        dropped += 1
            if not out:
                emptied += 1
            dst = f"{lbl_dir}/{os.path.basename(lf)}"
            with open(dst, "w") as f:
                f.write("\n".join(out))
                if out:
                    f.write("\n")
        print(f"  {split:6s}: kept {kept:7d} boxes, dropped {dropped:6d}, "
              f"{emptied} files now empty (background images — this is fine)")

    for c in glob.glob(f"{UNIFIED}/**/*.cache", recursive=True):
        os.remove(c)
        print("  removed stale cache:", os.path.basename(c))

    open(MARKER, "w").write("ok")
    print(f"remapped in {time.time()-t0:.0f}s")

MASTER = KEEP
NC = NC10

In [ ]:
bad, total, seen = 0, 0, set()
for split in ["train", "valid", "test"]:
    for lf in glob.glob(f"{UNIFIED}/{split}/labels/*.txt")[:5000]:
        with open(lf) as f:
            for line in f:
                parts = line.split()
                if not parts:
                    continue
                total += 1
                try:
                    c = int(parts[0])
                    vals = [float(x) for x in parts[1:5]]
                except ValueError:
                    bad += 1
                    print("MALFORMED:", lf, repr(line[:80]))
                    continue
                seen.add(c)
                if not (0 <= c < NC) or any(not (0.0 <= v <= 1.0) for v in vals):
                    bad += 1
                    print("OUT OF RANGE:", lf, repr(line[:80]))

print(f"checked {total} label lines, {bad} bad")
print(f"class ids present: {sorted(seen)}  (expected 0..{NC-1})")
assert bad == 0, "malformed labels — fix before training"
assert max(seen) < NC, f"class id {max(seen)} >= NC={NC}"
print("labels OK")

In [ ]:
b_hits = glob.glob("/kaggle/input/**/phase2_baseline.json", recursive=True) + \
         glob.glob(f"{WORK}/phase2_baseline.json")
phase2_baseline = None
if b_hits:
    with open(b_hits[0]) as f:
        phase2_baseline = json.load(f)
    print(json.dumps(phase2_baseline, indent=2))
else:
    print("phase2_baseline.json not found — will still re-score the baseline on test below")

In [ ]:
FAST_VAL = f"{WORK}/fast_val"

def build_fast_val():
    if os.path.exists(f"{FAST_VAL}/images") and len(os.listdir(f"{FAST_VAL}/images")) > 0:
        print("fast_val already built:", len(os.listdir(f"{FAST_VAL}/images")), "images")
        return

    img_dir, lbl_dir = f"{UNIFIED}/valid/images", f"{UNIFIED}/valid/labels"
    by_class = {i: [] for i in range(NC)}
    all_stems = []
    for lf in glob.glob(f"{lbl_dir}/*.txt"):
        stem = os.path.splitext(os.path.basename(lf))[0]
        all_stems.append(stem)
        with open(lf) as f:
            cs = {int(l.split()[0]) for l in f.read().strip().splitlines() if l.strip()}
        for c in cs:
            if c in by_class:
                by_class[c].append(stem)

    rng = random.Random(SEED)
    picked = set()
   
    for c in sorted(by_class, key=lambda k: len(by_class[k])):
        pool = [s for s in by_class[c] if s not in picked]
        rng.shuffle(pool)
        picked.update(pool[:250])
    
    rest = [s for s in all_stems if s not in picked]
    rng.shuffle(rest)
    picked.update(rest[:max(0, FAST_VAL_SIZE - len(picked))])

    os.makedirs(f"{FAST_VAL}/images", exist_ok=True)
    os.makedirs(f"{FAST_VAL}/labels", exist_ok=True)
    ext_map = {os.path.splitext(f)[0]: f for f in os.listdir(img_dir)}
    n = 0
    for stem in picked:
        if stem not in ext_map:
            continue
        shutil.copy2(f"{img_dir}/{ext_map[stem]}", f"{FAST_VAL}/images/{ext_map[stem]}")
        shutil.copy2(f"{lbl_dir}/{stem}.txt", f"{FAST_VAL}/labels/{stem}.txt")
        n += 1
    print(f"fast_val built: {n} images")

build_fast_val()


train_yaml = {
    "path": UNIFIED,
    "train": "train/images",
    "val": f"{FAST_VAL}/images",
    "test": "test/images",
    "nc": NC,
    "names": MASTER,
}
TRAIN_YAML = f"{WORK}/data_train.yaml"
with open(TRAIN_YAML, "w") as f:
    yaml.safe_dump(train_yaml, f, sort_keys=False)


eval_yaml = dict(train_yaml)
eval_yaml["val"] = "valid/images"
EVAL_YAML = f"{WORK}/data_eval.yaml"
with open(EVAL_YAML, "w") as f:
    yaml.safe_dump(eval_yaml, f, sort_keys=False)

print(open(TRAIN_YAML).read())

In [ ]:
if PROBE_ONLY:
    from ultralytics import YOLO
    print("=== TIMING PROBE (not the real run) ===")
    FRACTION = 0.04
    t0 = time.time()
    probe = YOLO(BASE_MODEL)
    probe.train(
        data=TRAIN_YAML, epochs=2, imgsz=IMGSZ, batch=BATCH, workers=WORKERS,
        device=DEVICE, fraction=FRACTION, seed=SEED, val=True, plots=False,
        project=f"{WORK}/probe", name="probe", exist_ok=True, verbose=True,
    )
    elapsed = time.time() - t0

    per_epoch_probe = elapsed / 2
   
    est_full_epoch_min = (per_epoch_probe / FRACTION) / 60
    print("\n" + "=" * 60)
    print(f"probe wall clock            : {elapsed/60:.1f} min for 2 epochs @ {FRACTION:.0%}")
    print(f"ESTIMATED full epoch        : ~{est_full_epoch_min:.1f} min")
    print(f"  (this OVERESTIMATES a bit — startup + val are counted at full weight)")
    for e in [30, 40, 50, 60, 80, 100]:
        h = e * est_full_epoch_min / 60
        flag = "OK" if h < TIME_BUDGET_HOURS else "TOO LONG"
        print(f"  {e:3d} epochs -> ~{h:5.1f} h   [{flag}]")
    print("=" * 60)
    print("\nPick an EPOCHS value marked OK, set PROBE_ONLY = False, and re-commit.")
else:
    print("PROBE_ONLY is False — skipping probe, going to the real run.")

In [ ]:
def find_resume_ckpt():
    local = f"{RUN_PROJECT}/{RUN_NAME}/weights/last.pt"
    if os.path.exists(local):
        return local
    hits = glob.glob("/kaggle/input/**/ppe_runs/**/weights/last.pt", recursive=True)
    if hits:
        
        dst_dir = f"{RUN_PROJECT}/{RUN_NAME}/weights"
        os.makedirs(dst_dir, exist_ok=True)
        src_run = os.path.dirname(os.path.dirname(hits[0]))
        for item in os.listdir(src_run):
            s, d = f"{src_run}/{item}", f"{RUN_PROJECT}/{RUN_NAME}/{item}"
            if os.path.isdir(s):
                shutil.copytree(s, d, dirs_exist_ok=True)
            else:
                shutil.copy2(s, d)
        return f"{dst_dir}/last.pt"
    return None

RESUME_CKPT = None if PROBE_ONLY else find_resume_ckpt()
if RESUME_CKPT:
    ck = torch.load(RESUME_CKPT, map_location="cpu", weights_only=False)
    print(f"RESUMING from {RESUME_CKPT} (completed epoch {ck.get('epoch')})")
else:
    print("No prior checkpoint — starting fresh.")

In [ ]:
if not PROBE_ONLY:
    from ultralytics import YOLO
    print("=== PRE-FLIGHT: 1 epoch, 1% of data, single GPU ===")
    try:
        _sm = YOLO(BASE_MODEL)
        _sm.train(data=TRAIN_YAML, epochs=1, imgsz=IMGSZ, batch=8, workers=2,
                  device=0, fraction=0.01, seed=SEED, val=False, plots=False,
                  project=f"{WORK}/smoke", name="smoke", exist_ok=True, verbose=False)
        print("\nPRE-FLIGHT PASSED — dataset and model load correctly.")
        SMOKE_OK = True
    except Exception as e:
        SMOKE_OK = False
        print("\nPRE-FLIGHT FAILED — this is the real error:")
        raise
    finally:
        shutil.rmtree(f"{WORK}/smoke", ignore_errors=True)
else:
    SMOKE_OK = False

In [ ]:
if not PROBE_ONLY:
    from ultralytics import YOLO

    t0 = time.time()

    def run_training(device):
        m = YOLO(BASE_MODEL)
        return m, m.train(
            data=TRAIN_YAML,
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=BATCH,
            workers=WORKERS,
            device=device,
            patience=PATIENCE,
            seed=SEED,
            project=RUN_PROJECT,
            name=RUN_NAME,
            exist_ok=True,

          
            save_period=5,
            time=TIME_BUDGET_HOURS,

           
            optimizer="AdamW",
            lr0=0.001,
            lrf=0.01,
            cos_lr=True,
            warmup_epochs=3.0,
            weight_decay=0.0005,

            
            cls=0.7,
            box=7.5,
            dfl=1.5,

           
            mosaic=0.5,   
            close_mosaic=10,
            degrees=5.0,
            fliplr=0.5,
            scale=0.5,
            translate=0.1,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
            erasing=0.4,

            plots=True,
            verbose=True,
        )

    if RESUME_CKPT:
        model = YOLO(RESUME_CKPT)
        results = model.train(resume=True)
    else:
        try:
            model, results = run_training(DEVICE)
        except Exception as e:
           
            print("=" * 70)
            print("MULTI-GPU (DDP) RUN FAILED:", type(e).__name__, e)
            print("Falling back to a single GPU. Check the Logs tab for the real")
            print("error from the subprocess if you want the details.")
            print("=" * 70)
            import shutil as _sh
            print("disk free in /kaggle/working: "
                  f"{_sh.disk_usage(WORK).free/1e9:.1f} GB")
            BATCH_1GPU = 16
            print(f"retrying: device=0, batch={BATCH_1GPU}")
            globals()["BATCH"] = BATCH_1GPU
            model, results = run_training(0)

    print(f"\nTRAINING DONE in {(time.time()-t0)/3600:.2f} h")
else:
    print("PROBE_ONLY is True — skipping training.")

In [ ]:
from ultralytics import YOLO

BEST = f"{RUN_PROJECT}/{RUN_NAME}/weights/best.pt"
assert os.path.exists(BEST), f"no best.pt at {BEST} — did training run?"
custom = YOLO(BEST)

def per_class_frame(res, names):
    rows = []
    for i, c in enumerate(res.box.ap_class_index):
        rows.append({
            "class": names[int(c)],
            "P":        float(res.box.p[i]),
            "R":        float(res.box.r[i]),
            "mAP50":    float(res.box.ap50[i]),
            "mAP50_95": float(res.box.ap[i]),
        })
    return pd.DataFrame(rows)

print("### CUSTOM MODEL on TEST (12 classes) ###")
custom_test = custom.val(data=EVAL_YAML, split="test", imgsz=IMGSZ,
                         batch=16, device=0, plots=True, verbose=True)
custom_test_df = per_class_frame(custom_test, MASTER)
print(custom_test_df.to_string(index=False))
print(f"\noverall mAP50 = {custom_test.box.map50:.4f} | mAP50-95 = {custom_test.box.map:.4f}")

In [ ]:
print("### CUSTOM MODEL on full VALID (12 classes) ###")
custom_valid = custom.val(data=EVAL_YAML, split="val", imgsz=IMGSZ,
                          batch=16, device=0, plots=False, verbose=True)
custom_valid_df = per_class_frame(custom_valid, MASTER)
print(f"\noverall mAP50 = {custom_valid.box.map50:.4f} | mAP50-95 = {custom_valid.box.map:.4f}")

In [ ]:
!git clone -q https://github.com/snehilsanyal/Construction-Site-Safety-PPE-Detection.git {WORK}/model10_repo 2>/dev/null || true

BASE_PT = f"{WORK}/model10_repo/models/best.pt"

ORIG_CLASSES = ["Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
                "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle"]

BASE_TEST = f"{WORK}/test_baseline10"
os.makedirs(f"{BASE_TEST}/images", exist_ok=True)
os.makedirs(f"{BASE_TEST}/labels", exist_ok=True)

ORIG_LBL = f"{UNIFIED}/test/labels_orig12"   
n = 0
for fname in os.listdir(f"{UNIFIED}/test/images"):
    stem = os.path.splitext(fname)[0]
    lf = f"{ORIG_LBL}/{stem}.txt"
    if not os.path.exists(lf):
        continue
    with open(lf) as f:
        lines = [l for l in f.read().strip().splitlines()
                 if l.strip() and int(l.split()[0]) < 10]
    shutil.copy2(f"{UNIFIED}/test/images/{fname}", f"{BASE_TEST}/images/{fname}")
    with open(f"{BASE_TEST}/labels/{stem}.txt", "w") as f:
        f.write("\n".join(lines))
    n += 1
print("baseline test images:", n)

with open(f"{BASE_TEST}/data.yaml", "w") as f:
    yaml.safe_dump({"path": BASE_TEST, "train": "images", "val": "images",
                    "nc": 10, "names": ORIG_CLASSES}, f, sort_keys=False)

print("\n### PHASE-1 BASELINE on TEST (10 classes) ###")
baseline_model = YOLO(BASE_PT)
base_test = baseline_model.val(data=f"{BASE_TEST}/data.yaml", split="val",
                               imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=True)
base_test_df = per_class_frame(base_test, ORIG_CLASSES)
print(f"\noverall mAP50 = {base_test.box.map50:.4f} | mAP50-95 = {base_test.box.map:.4f}")

In [ ]:
SHARED = [c for c in MASTER if c in ORIG_CLASSES]
SHARED_IDX = [MASTER.index(c) for c in SHARED]
print(f"### CUSTOM MODEL on TEST, restricted to the {len(SHARED)} shared classes ###")
print("shared:", SHARED)
custom_test_10 = custom.val(data=EVAL_YAML, split="test", imgsz=IMGSZ, batch=16,
                            device=0, classes=SHARED_IDX, plots=False, verbose=False)
print(f"mAP50 = {custom_test_10.box.map50:.4f} | mAP50-95 = {custom_test_10.box.map:.4f}")

In [ ]:
comparison = pd.DataFrame([
    {"model": "Phase-1 pretrained (snehilsanyal best.pt)",
     "classes": 10, "split": "test",
     "mAP50": float(base_test.box.map50),
     "mAP50_95": float(base_test.box.map),
     "precision": float(base_test.box.mp),
     "recall": float(base_test.box.mr)},
    {"model": f"Phase-3 v2 custom ({BASE_MODEL}) — shared classes only",
     "classes": len(SHARED), "split": "test",
     "mAP50": float(custom_test_10.box.map50),
     "mAP50_95": float(custom_test_10.box.map),
     "precision": float(custom_test_10.box.mp),
     "recall": float(custom_test_10.box.mr)},
    {"model": f"Phase-3 v2 custom ({BASE_MODEL}) — full 10 classes",
     "classes": NC, "split": "test",
     "mAP50": float(custom_test.box.map50),
     "mAP50_95": float(custom_test.box.map),
     "precision": float(custom_test.box.mp),
     "recall": float(custom_test.box.mr)},
]).round(4)

print(comparison.to_string(index=False))

delta50 = float(custom_test_10.box.map50) - float(base_test.box.map50)
print(f"\nHEADLINE: custom model beats the Phase-1 baseline on the held-out test set "
      f"by {delta50:+.4f} mAP@0.5 ({delta50/max(1e-9, float(base_test.box.map50)):+.1%} relative).")

comparison.to_csv(f"{WORK}/phase3_comparison.csv", index=False)

In [ ]:
merged = base_test_df.merge(custom_test_df, on="class", suffixes=("_baseline", "_custom"))
merged["mAP50_delta"] = (merged["mAP50_custom"] - merged["mAP50_baseline"]).round(4)
merged = merged.round(4).sort_values("mAP50_delta")
print(merged[["class", "mAP50_baseline", "mAP50_custom", "mAP50_delta"]].to_string(index=False))
merged.to_csv(f"{WORK}/phase3_per_class_comparison.csv", index=False)

In [ ]:
weak = custom_test_df.sort_values("mAP50").head(5)
print("WEAKEST 5 CLASSES on test:")
print(weak.to_string(index=False))

print("\nDiagnosis per weak class:")
test_counts = Counter()
for lf in glob.glob(f"{UNIFIED}/test/labels/*.txt"):
    with open(lf) as f:
        for l in f.read().strip().splitlines():
            if l.strip():
                test_counts[int(l.split()[0])] += 1

train_counts = Counter()
for lf in glob.glob(f"{UNIFIED}/train/labels/*.txt"):
    with open(lf) as f:
        for l in f.read().strip().splitlines():
            if l.strip():
                train_counts[int(l.split()[0])] += 1

for _, row in weak.iterrows():
    ci = MASTER.index(row["class"])
    p, r = row["P"], row["R"]
    if r < 0.4 and p > 0.6:
        mode = "MISSES objects (low recall, decent precision) -> needs more/harder positives"
    elif p < 0.4 and r > 0.6:
        mode = "OVER-PREDICTS (low precision) -> confusing this class with a visually similar one"
    else:
        mode = "weak on both -> genuinely under-represented or ambiguous labels"
    print(f"  {row['class']:16s} train={train_counts.get(ci,0):6d}  test={test_counts.get(ci,0):5d}  "
          f"P={p:.2f} R={r:.2f}  |  {mode}")

In [ ]:
from IPython.display import Image as IPyImage, display

run_dir = f"{RUN_PROJECT}/{RUN_NAME}"
for plot in ["results.png", "confusion_matrix_normalized.png", "PR_curve.png",
             "F1_curve.png", "labels.jpg"]:
    p = f"{run_dir}/{plot}"
    if os.path.exists(p):
        print("\n===", plot, "===")
        display(IPyImage(filename=p, width=900))

In [ ]:
csvp = f"{run_dir}/results.csv"
if os.path.exists(csvp):
    hist = pd.read_csv(csvp)
    hist.columns = [c.strip() for c in hist.columns]
    print(hist.tail(10).to_string(index=False))
    best_row = hist.loc[hist["metrics/mAP50(B)"].idxmax()]
    print(f"\nbest epoch: {int(best_row['epoch'])}  "
          f"mAP50={best_row['metrics/mAP50(B)']:.4f}  "
          f"mAP50-95={best_row['metrics/mAP50-95(B)']:.4f}")

In [ ]:
artifacts = {
    "phase": 3,
    "config": cfg,
    "dataset": {
        "train_images": len(os.listdir(f"{UNIFIED}/train/images")),
        "valid_images": len(os.listdir(f"{UNIFIED}/valid/images")),
        "test_images":  len(os.listdir(f"{UNIFIED}/test/images")),
        "classes": MASTER,
    },
    "custom_model": {
        "checkpoint": BEST,
        "test_12class":  {"mAP50": float(custom_test.box.map50),
                          "mAP50_95": float(custom_test.box.map),
                          "precision": float(custom_test.box.mp),
                          "recall": float(custom_test.box.mr)},
        "test_10class":  {"mAP50": float(custom_test_10.box.map50),
                          "mAP50_95": float(custom_test_10.box.map)},
        "valid_12class": {"mAP50": float(custom_valid.box.map50),
                          "mAP50_95": float(custom_valid.box.map)},
        "per_class_test": custom_test_df.to_dict(orient="records"),
    },
    "phase1_baseline": {
        "model": "snehilsanyal best.pt",
        "test_10class": {"mAP50": float(base_test.box.map50),
                         "mAP50_95": float(base_test.box.map),
                         "precision": float(base_test.box.mp),
                         "recall": float(base_test.box.mr)},
        "per_class_test": base_test_df.to_dict(orient="records"),
    },
    "improvement_mAP50_10class": delta50,
    "phase2_baseline_reference": phase2_baseline,
}

with open(f"{WORK}/phase3_results.json", "w") as f:
    json.dump(artifacts, f, indent=2)


shutil.copy2(BEST, f"{WORK}/phase3_best.pt")
!cd {WORK} && zip -qr phase3_run_logs.zip ppe_runs -x "*/weights/epoch*.pt"

print("saved:")
for f_ in ["phase3_results.json", "phase3_comparison.csv",
           "phase3_per_class_comparison.csv", "phase3_best.pt", "phase3_run_logs.zip"]:
    p = f"{WORK}/{f_}"
    if os.path.exists(p):
        print(f"  {f_:36s} {os.path.getsize(p)/1e6:8.2f} MB")

In [ ]:
for d in [UNIFIED, FAST_VAL, BASE_TEST, f"{WORK}/model10_repo", f"{WORK}/probe"]:
    if os.path.exists(d):
        shutil.rmtree(d, ignore_errors=True)
for z in glob.glob(f"{WORK}/*.zip"):
    if "phase3" not in os.path.basename(z):
        os.remove(z)
print("cleaned. remaining in /kaggle/working:")
for f_ in sorted(os.listdir(WORK)):
    p = f"{WORK}/{f_}"
    sz = os.path.getsize(p)/1e6 if os.path.isfile(p) else -1
    print(f"  {f_:40s} {sz:8.2f} MB" if sz >= 0 else f"  {f_}/")